### PyTorch Tensors

In [1]:
import torch

In [2]:
X = torch.tensor([[1.0, 4.0, 7.0], [2.0, 3.0, 6.0]])
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [3]:
X.shape

torch.Size([2, 3])

In [4]:
X.dtype

torch.float32

In [5]:
X[0, 1]

tensor(4.)

In [6]:
X[:, 1]

tensor([4., 3.])

In [7]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [8]:
X.exp()

tensor([[   2.7183,   54.5982, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])

In [9]:
X.mean()

tensor(3.8333)

In [10]:
X.max(dim=0)

torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))

In [11]:
X @ X.T

tensor([[66., 56.],
        [56., 49.]])

In [12]:
import numpy as np

X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [13]:
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [14]:
torch.tensor(np.array([[1., 4., 7.], [2., 3., 6.]]), dtype=torch.float32)

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [15]:
torch.FloatTensor(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [16]:
torch.from_numpy(np.array([[1., 4., 7.], [2., 3., 6.]]))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

In [17]:
X[:, 1] = -99
X


tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In [18]:
X.relu_()
X

tensor([[1., 0., 7.],
        [2., 0., 6.]])

### Hardware Acceleration

In [19]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'mps'

In [20]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)

In [21]:
M.device

device(type='mps', index=0)

In [22]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

In [23]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='mps:0')

In [24]:
M = torch.rand(1000, 1000)
%timeit M @ M.T

761 μs ± 1.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [25]:
M = torch.rand(1000, 1000, device="mps")
%timeit M @ M.T

242 μs ± 43.2 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


The `timeit` average is about 5 times lower with the GPU, but the overall (clock) time is more than double. Perhaps because the bottleneck is moving data in and out of the GPU.

### Autograd

In [26]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [27]:
f.backward()
x.grad

tensor(10.)

In [28]:
# gradient descent step (disabling gradient tracking)
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad
x

tensor(4., requires_grad=True)

In [29]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad
x_detached

tensor(3.)

In [30]:
# modifying x_detached also modifies x
x

tensor(3., requires_grad=True)

In [31]:
x.grad.zero_()

tensor(0.)

In [32]:
# Training loop
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2  # forward pass
    f.backward()  # backward pass, computes the gradients
    with torch.no_grad():
        x -= learning_rate * x.grad  # gradient descent step
    
    x.grad.zero_()

x

tensor(1.0185e-09, requires_grad=True)

In [33]:
t = torch.tensor(2.0, requires_grad=True)
z = t.exp()  # this is an intermediate result
z +=1  # in-place operation
# z.backward()  # RuntimeError

In [34]:
t = torch.tensor(2.0, requires_grad=True)
z = t.exp()  # this is an intermediate result
z = z + 1  # remove in-place operation
z.backward()

### Linear Regression

In [35]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

In [36]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [37]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [38]:
torch.manual_seed(42)
n_features = X_train.shape[1]  # 8 input features
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [39]:
# Batch Gradient Descent using autograd
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {loss.item()}")

Epoch 1 / 20, Loss: 16.158458709716797
Epoch 2 / 20, Loss: 4.87937593460083
Epoch 3 / 20, Loss: 2.255227565765381
Epoch 4 / 20, Loss: 1.330764651298523
Epoch 5 / 20, Loss: 0.9680710434913635
Epoch 6 / 20, Loss: 0.8142688870429993
Epoch 7 / 20, Loss: 0.7417054176330566
Epoch 8 / 20, Loss: 0.702070951461792
Epoch 9 / 20, Loss: 0.6765925288200378
Epoch 10 / 20, Loss: 0.65779709815979
Epoch 11 / 20, Loss: 0.6426157355308533
Epoch 12 / 20, Loss: 0.6297228336334229
Epoch 13 / 20, Loss: 0.6184946298599243
Epoch 14 / 20, Loss: 0.6085972189903259
Epoch 15 / 20, Loss: 0.5998220443725586
Epoch 16 / 20, Loss: 0.5920190215110779
Epoch 17 / 20, Loss: 0.5850694179534912
Epoch 18 / 20, Loss: 0.5788736343383789
Epoch 19 / 20, Loss: 0.5733456015586853
Epoch 20 / 20, Loss: 0.5684102773666382


In [40]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = X_new @ w + b
y_pred

tensor([[0.8916],
        [1.6480],
        [2.6577]])

### Linear Regression using High-Level API

In [41]:
import torch.nn as nn   # Convention
torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [42]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [43]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [44]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [45]:
model(X_train[:2])

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

In [46]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [47]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {loss.item()}")

In [48]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1 / 20, Loss: 4.3378496170043945
Epoch 2 / 20, Loss: 0.7802932262420654
Epoch 3 / 20, Loss: 0.6253839731216431
Epoch 4 / 20, Loss: 0.6060433983802795
Epoch 5 / 20, Loss: 0.595629870891571
Epoch 6 / 20, Loss: 0.5873565673828125
Epoch 7 / 20, Loss: 0.5802989602088928
Epoch 8 / 20, Loss: 0.5741382241249084
Epoch 9 / 20, Loss: 0.5687100291252136
Epoch 10 / 20, Loss: 0.5639079213142395
Epoch 11 / 20, Loss: 0.5596510767936707
Epoch 12 / 20, Loss: 0.5558737516403198
Epoch 13 / 20, Loss: 0.5525193810462952
Epoch 14 / 20, Loss: 0.5495391488075256
Epoch 15 / 20, Loss: 0.5468899011611938
Epoch 16 / 20, Loss: 0.544533908367157
Epoch 17 / 20, Loss: 0.5424376726150513
Epoch 18 / 20, Loss: 0.5405715703964233
Epoch 19 / 20, Loss: 0.5389097332954407
Epoch 20 / 20, Loss: 0.5374287962913513


In [49]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)

y_pred

tensor([[0.8061],
        [1.7116],
        [2.6973]])

### Regression LMP

In [50]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [51]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1 / 20, Loss: 5.045480251312256
Epoch 2 / 20, Loss: 2.0523128509521484
Epoch 3 / 20, Loss: 1.00398850440979
Epoch 4 / 20, Loss: 0.8570139408111572
Epoch 5 / 20, Loss: 0.7740675210952759
Epoch 6 / 20, Loss: 0.7225848436355591
Epoch 7 / 20, Loss: 0.6893726587295532
Epoch 8 / 20, Loss: 0.6669033169746399
Epoch 9 / 20, Loss: 0.6507739424705505
Epoch 10 / 20, Loss: 0.6383934020996094
Epoch 11 / 20, Loss: 0.6281994581222534
Epoch 12 / 20, Loss: 0.6193398833274841
Epoch 13 / 20, Loss: 0.6113173365592957
Epoch 14 / 20, Loss: 0.6038705110549927
Epoch 15 / 20, Loss: 0.5968307852745056
Epoch 16 / 20, Loss: 0.5901119112968445
Epoch 17 / 20, Loss: 0.583646833896637
Epoch 18 / 20, Loss: 0.5774063467979431
Epoch 19 / 20, Loss: 0.5713555216789246
Epoch 20 / 20, Loss: 0.565444827079773


### Mini-Batch GD

In [52]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [53]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)
# model.to(device)

In [54]:
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [55]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")

In [56]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1 / 20, Loss: 0.5900
Epoch 2 / 20, Loss: 0.4046
Epoch 3 / 20, Loss: 0.3801
Epoch 4 / 20, Loss: 0.3629
Epoch 5 / 20, Loss: 0.3529
Epoch 6 / 20, Loss: 0.3520
Epoch 7 / 20, Loss: 0.3408
Epoch 8 / 20, Loss: 0.3427
Epoch 9 / 20, Loss: 0.3406
Epoch 10 / 20, Loss: 0.3378
Epoch 11 / 20, Loss: 0.3304
Epoch 12 / 20, Loss: 0.3267
Epoch 13 / 20, Loss: 0.3244
Epoch 14 / 20, Loss: 0.3221
Epoch 15 / 20, Loss: 0.3186
Epoch 16 / 20, Loss: 0.3149
Epoch 17 / 20, Loss: 0.3123
Epoch 18 / 20, Loss: 0.3111
Epoch 19 / 20, Loss: 0.3088
Epoch 20 / 20, Loss: 0.3072


### Evaluation

In [57]:
def evaluate(model, data_loader, metric_fn, aggregate_fn = torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

In [58]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.4080)

In [59]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

evaluate(model, valid_loader, rmse)

tensor(0.5668)

In [60]:
valid_mse.sqrt()

tensor(0.6388)

In [61]:
evaluate(model, valid_loader, mse, 
         aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.6388)

In [62]:
import torchmetrics

def evaluate_tm(model, data_loader, metric, device=None):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [63]:
rmse = torchmetrics.MeanSquaredError(squared=False)
evaluate_tm(model, valid_loader, rmse)

tensor(0.6388)

In [64]:
def train(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs, device=None):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_metric = evaluate_tm(model, train_loader, eval_metric, device)
        valid_metric = evaluate_tm(model, valid_loader, eval_metric, device)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train metric: {train_metric.item():.3f}, Valid metric: {valid_metric.item():.3f}")

In [65]:
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 0.3046
Train metric: 0.541, Valid metric: 0.632
Epoch 2 / 20, Loss: 0.2998
Train metric: 0.539, Valid metric: 0.580
Epoch 3 / 20, Loss: 0.2994
Train metric: 0.535, Valid metric: 0.584
Epoch 4 / 20, Loss: 0.2976
Train metric: 0.552, Valid metric: 0.576
Epoch 5 / 20, Loss: 0.2960
Train metric: 0.535, Valid metric: 0.533
Epoch 6 / 20, Loss: 0.2924
Train metric: 0.538, Valid metric: 0.587
Epoch 7 / 20, Loss: 0.2910
Train metric: 0.527, Valid metric: 0.545
Epoch 8 / 20, Loss: 0.2891
Train metric: 0.537, Valid metric: 0.558
Epoch 9 / 20, Loss: 0.2969
Train metric: 0.552, Valid metric: 0.738
Epoch 10 / 20, Loss: 0.2939
Train metric: 0.530, Valid metric: 0.871
Epoch 11 / 20, Loss: 0.2922
Train metric: 0.534, Valid metric: 0.536
Epoch 12 / 20, Loss: 0.2863
Train metric: 0.605, Valid metric: 0.626
Epoch 13 / 20, Loss: 0.2867
Train metric: 0.543, Valid metric: 0.558
Epoch 14 / 20, Loss: 0.2851
Train metric: 0.527, Valid metric: 0.597
Epoch 15 / 20, Loss: 0.2827
Train metric: 0

### Nonsequential Models

In [66]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features, 1)  # 40 + n_features due to concat

    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)


In [67]:
torch.manual_seed(42)
model = WideAndDeep(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4093
Train metric: 0.797, Valid metric: 0.879
Epoch 2 / 20, Loss: 0.6084
Train metric: 0.761, Valid metric: 0.793
Epoch 3 / 20, Loss: 0.5648
Train metric: 0.739, Valid metric: 0.873
Epoch 4 / 20, Loss: 0.5270
Train metric: 0.716, Valid metric: 0.795
Epoch 5 / 20, Loss: 0.5048
Train metric: 0.700, Valid metric: 0.705
Epoch 6 / 20, Loss: 0.4810
Train metric: 0.687, Valid metric: 0.699
Epoch 7 / 20, Loss: 0.4642
Train metric: 0.676, Valid metric: 0.688
Epoch 8 / 20, Loss: 0.4501
Train metric: 0.666, Valid metric: 0.643
Epoch 9 / 20, Loss: 0.4396
Train metric: 0.659, Valid metric: 0.664
Epoch 10 / 20, Loss: 0.4301
Train metric: 0.652, Valid metric: 0.651
Epoch 11 / 20, Loss: 0.4222
Train metric: 0.647, Valid metric: 0.645
Epoch 12 / 20, Loss: 0.4157
Train metric: 0.642, Valid metric: 0.645
Epoch 13 / 20, Loss: 0.4105
Train metric: 0.638, Valid metric: 0.620
Epoch 14 / 20, Loss: 0.4044
Train metric: 0.633, Valid metric: 0.735
Epoch 15 / 20, Loss: 0.4007
Train metric: 0

In [68]:
class WideAndDeepV2(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)

    def forward(self, X):
        X_wide = X[:, :5]
        X_deep = X[:, 2:]
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [69]:
torch.manual_seed(42)
model = WideAndDeepV2(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, valid_loader, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4730
Train metric: 0.803, Valid metric: 0.988
Epoch 2 / 20, Loss: 0.5910
Train metric: 0.744, Valid metric: 0.756
Epoch 3 / 20, Loss: 0.5329
Train metric: 0.716, Valid metric: 0.706
Epoch 4 / 20, Loss: 0.4996
Train metric: 0.697, Valid metric: 0.685
Epoch 5 / 20, Loss: 0.4776
Train metric: 0.684, Valid metric: 0.671
Epoch 6 / 20, Loss: 0.4630
Train metric: 0.676, Valid metric: 0.667
Epoch 7 / 20, Loss: 0.4532
Train metric: 0.669, Valid metric: 0.663
Epoch 8 / 20, Loss: 0.4461
Train metric: 0.665, Valid metric: 0.662
Epoch 9 / 20, Loss: 0.4405
Train metric: 0.662, Valid metric: 0.669
Epoch 10 / 20, Loss: 0.4358
Train metric: 0.658, Valid metric: 0.667
Epoch 11 / 20, Loss: 0.4317
Train metric: 0.656, Valid metric: 0.671
Epoch 12 / 20, Loss: 0.4293
Train metric: 0.653, Valid metric: 0.675
Epoch 13 / 20, Loss: 0.4248
Train metric: 0.651, Valid metric: 0.675
Epoch 14 / 20, Loss: 0.4231
Train metric: 0.648, Valid metric: 0.676
Epoch 15 / 20, Loss: 0.4201
Train metric: 0

### Multiple Inputs

In [70]:
class WideAndDeepV3(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [71]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
valid_data_wd = TensorDataset(X_valid[:, :5], X_valid[:, 2:], y_valid)
valid_loader_wd = DataLoader(valid_data_wd, batch_size=32, shuffle=True)
test_data_wd = TensorDataset(X_test[:, :5], X_test[:, 2:], y_test)
test_loader_wd = DataLoader(test_data_wd, batch_size=32, shuffle=True)

In [72]:
def evaluate_multiple_inputs(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        # for X_batch_wide, X_batch_deep, y_batch in data_loader:
        for *X_batch_inputs, y_batch in data_loader:
            # y_pred = model(X_batch_wide, X_batch_deep)
            y_pred = model(*X_batch_inputs)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train_multiple_inputs(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        # for X_batch_wide, X_batch_deep, y_batch in train_loader:
        for *X_batch_inputs, y_batch in train_loader:
            y_pred = model(*X_batch_inputs)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_rmse = evaluate_multiple_inputs(model, train_loader, eval_metric)
        valid_rmse = evaluate_multiple_inputs(model, valid_loader, eval_metric)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train rmse: {train_rmse.item():.3f}, Valid rmse: {valid_rmse.item():.3f}")

In [73]:
torch.manual_seed(42)
model = WideAndDeepV3(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_multiple_inputs(model, optimizer, mse, train_loader_wd, valid_loader_wd, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.4730
Train rmse: 0.803, Valid rmse: 0.988
Epoch 2 / 20, Loss: 0.5921
Train rmse: 0.744, Valid rmse: 0.759
Epoch 3 / 20, Loss: 0.5324
Train rmse: 0.716, Valid rmse: 0.708
Epoch 4 / 20, Loss: 0.4996
Train rmse: 0.698, Valid rmse: 0.686
Epoch 5 / 20, Loss: 0.4778
Train rmse: 0.685, Valid rmse: 0.672
Epoch 6 / 20, Loss: 0.4632
Train rmse: 0.676, Valid rmse: 0.665
Epoch 7 / 20, Loss: 0.4532
Train rmse: 0.669, Valid rmse: 0.663
Epoch 8 / 20, Loss: 0.4454
Train rmse: 0.665, Valid rmse: 0.665
Epoch 9 / 20, Loss: 0.4407
Train rmse: 0.662, Valid rmse: 0.665
Epoch 10 / 20, Loss: 0.4364
Train rmse: 0.659, Valid rmse: 0.667
Epoch 11 / 20, Loss: 0.4311
Train rmse: 0.656, Valid rmse: 0.670
Epoch 12 / 20, Loss: 0.4287
Train rmse: 0.654, Valid rmse: 0.670
Epoch 13 / 20, Loss: 0.4261
Train rmse: 0.651, Valid rmse: 0.671
Epoch 14 / 20, Loss: 0.4232
Train rmse: 0.649, Valid rmse: 0.674
Epoch 15 / 20, Loss: 0.4207
Train rmse: 0.649, Valid rmse: 0.673
Epoch 16 / 20, Loss: 0.4186
Train 

### Multiple Outputs

In [74]:
class WideAndDeepV4(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features - 2, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        self.output_layer = nn.Linear(40 + n_features - 3, 1)
        self.aux_output_layer = nn.Linear(40, 1)

    def forward(self, X_wide, X_deep):
        deep_output = self.deep_stack(X_deep)
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
        main_output = self.output_layer(wide_and_deep)
        aux_output = self.aux_output_layer(deep_output)
        return main_output, aux_output

In [75]:
def evaluate_v4(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for *X_batch_inputs, y_batch in data_loader:
            y_pred, _ = model(*X_batch_inputs)  # ignore auxiliary output
            metric.update(y_pred, y_batch)
    return metric.compute()

def train_v4(model, optimizer, criterion, train_loader, valid_loader, eval_metric, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.
        for *X_batch_inputs, y_batch in train_loader:
            y_pred, y_pred_aux = model(*X_batch_inputs)
            main_loss = criterion(y_pred, y_batch)
            aux_loss = criterion(y_pred_aux, y_batch)
            loss = 0.8 * main_loss + 0.2 * aux_loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_loss = total_loss / len(train_loader)
        train_rmse = evaluate_v4(model, train_loader, eval_metric)
        valid_rmse = evaluate_v4(model, valid_loader, eval_metric)
        print(f"Epoch {epoch + 1} / {n_epochs}, Loss: {mean_loss:.4f}")
        print(f"Train rmse: {train_rmse.item():.3f}, Valid rmse: {valid_rmse.item():.3f}")

In [76]:
torch.manual_seed(42)
model = WideAndDeepV4(n_features)
learning_rate = 0.002
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_v4(model, optimizer, mse, train_loader_wd, valid_loader_wd, rmse, n_epochs)

Epoch 1 / 20, Loss: 1.9046
Train rmse: 0.836, Valid rmse: 1.250
Epoch 2 / 20, Loss: 0.7666
Train rmse: 0.764, Valid rmse: 0.837
Epoch 3 / 20, Loss: 0.6790
Train rmse: 0.733, Valid rmse: 0.726
Epoch 4 / 20, Loss: 0.6339
Train rmse: 0.712, Valid rmse: 0.694
Epoch 5 / 20, Loss: 0.6009
Train rmse: 0.696, Valid rmse: 0.677
Epoch 6 / 20, Loss: 0.5756
Train rmse: 0.685, Valid rmse: 0.667
Epoch 7 / 20, Loss: 0.5555
Train rmse: 0.677, Valid rmse: 0.659
Epoch 8 / 20, Loss: 0.5399
Train rmse: 0.671, Valid rmse: 0.658
Epoch 9 / 20, Loss: 0.5267
Train rmse: 0.667, Valid rmse: 0.658
Epoch 10 / 20, Loss: 0.5161
Train rmse: 0.664, Valid rmse: 0.660
Epoch 11 / 20, Loss: 0.5065
Train rmse: 0.661, Valid rmse: 0.667
Epoch 12 / 20, Loss: 0.4979
Train rmse: 0.658, Valid rmse: 0.670
Epoch 13 / 20, Loss: 0.4902
Train rmse: 0.655, Valid rmse: 0.675
Epoch 14 / 20, Loss: 0.4837
Train rmse: 0.653, Valid rmse: 0.680
Epoch 15 / 20, Loss: 0.4773
Train rmse: 0.650, Valid rmse: 0.683
Epoch 16 / 20, Loss: 0.4716
Train 

### Building an Image Classifier with PyTorch

In [77]:
# Get the dataset with torchvision
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000]
)

In [78]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [79]:
X_sample, y_sample = train_data[0]
X_sample.shape  # channel, height, width

torch.Size([1, 28, 28])

In [80]:
X_sample.dtype

torch.float32

In [81]:
y_sample

9

In [83]:
train_and_valid_data.classes[y_sample]

'Ankle boot'

In [84]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )

    def forward(self, X):
        return self.mlp(X)
    
torch.manual_seed(42)
model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10)

xentropy = nn.CrossEntropyLoss()

In [85]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10) #.to(device)
train(model, optimizer, xentropy, train_loader, valid_loader, accuracy, n_epochs)

Epoch 1 / 20, Loss: 0.6060
Train metric: 0.853, Valid metric: 0.841
Epoch 2 / 20, Loss: 0.4036
Train metric: 0.866, Valid metric: 0.849


KeyboardInterrupt: 

In [84]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
model = model.to(device)
train(model, optimizer, xentropy, train_loader, valid_loader, accuracy, n_epochs, device=device)

Epoch 1 / 20, Loss: 0.1830
Train metric: 0.938, Valid metric: 0.889
Epoch 2 / 20, Loss: 0.1800
Train metric: 0.938, Valid metric: 0.892
Epoch 3 / 20, Loss: 0.1722
Train metric: 0.926, Valid metric: 0.880
Epoch 4 / 20, Loss: 0.1704
Train metric: 0.940, Valid metric: 0.889
Epoch 5 / 20, Loss: 0.1652
Train metric: 0.937, Valid metric: 0.888
Epoch 6 / 20, Loss: 0.1614
Train metric: 0.942, Valid metric: 0.892
Epoch 7 / 20, Loss: 0.1563
Train metric: 0.940, Valid metric: 0.884
Epoch 8 / 20, Loss: 0.1525
Train metric: 0.931, Valid metric: 0.881
Epoch 9 / 20, Loss: 0.1462
Train metric: 0.939, Valid metric: 0.888
Epoch 10 / 20, Loss: 0.1462
Train metric: 0.945, Valid metric: 0.884
Epoch 11 / 20, Loss: 0.1431
Train metric: 0.954, Valid metric: 0.889
Epoch 12 / 20, Loss: 0.1380
Train metric: 0.931, Valid metric: 0.876
Epoch 13 / 20, Loss: 0.1334
Train metric: 0.945, Valid metric: 0.889
Epoch 14 / 20, Loss: 0.1315
Train metric: 0.951, Valid metric: 0.880
Epoch 15 / 20, Loss: 0.1331
Train metric: 0

Still terribly slow on the GPU... :(

In [85]:
# Make predictions
model = model.to('cpu')
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3]
with torch.no_grad():
    y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)
y_pred

tensor([7, 4, 6])

In [86]:
[train_and_valid_data.classes[index] for index in y_pred]

['Sneaker', 'Coat', 'Shirt']

In [87]:
[train_and_valid_data.classes[index] for index in y_new[:3]]

['Sneaker', 'Coat', 'Pullover']

In [88]:
# Class probabilities
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.1190, 0.0000, 0.8750, 0.0000, 0.0060, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.2270, 0.0000, 0.0100, 0.0000, 0.7630, 0.0000, 0.0000,
         0.0000]])

In [89]:
y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_logits, dim=1)
y_top4_probas.round(decimals=3)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.8750, 0.1190, 0.0060, 0.0000],
        [0.7630, 0.2270, 0.0100, 0.0000]])

In [90]:
y_top4_indices

tensor([[7, 9, 1, 5],
        [4, 2, 6, 3],
        [6, 2, 4, 0]])

### Hyperparameter tuning with Optuna

In [94]:
def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

In [92]:
import optuna
from optuna.trial import Trial

def objective(trial: Trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden, n_hidden2=n_hidden,
                            n_classes=10)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    history = train2(model, optimizer, xentropy, accuracy, train_loader,
                     valid_loader, n_epochs=10)
    validation_accuracy = max(history["valid_metrics"])
    return validation_accuracy

In [95]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=5)

[I 2026-01-26 18:33:01,149] A new study created in memory with name: no-name-9a3702ef-66d7-4b6a-84bd-3b1a7cee40c7


Epoch 1/10, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1862
Epoch 2/10, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500
Epoch 3/10, train loss: 2.1164, train metric: 0.4110, valid metric: 0.4554
Epoch 4/10, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560
Epoch 5/10, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026
Epoch 6/10, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228
Epoch 7/10, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326
Epoch 8/10, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372
Epoch 9/10, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6424


[I 2026-01-26 18:33:30,630] Trial 0 finished with value: 0.6435999870300293 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.6435999870300293.


Epoch 10/10, train loss: 1.0782, train metric: 0.6537, valid metric: 0.6436
Epoch 1/10, train loss: 1.1459, train metric: 0.6229, valid metric: 0.7338
Epoch 2/10, train loss: 0.6108, train metric: 0.7842, valid metric: 0.7996
Epoch 3/10, train loss: 0.5203, train metric: 0.8170, valid metric: 0.8088
Epoch 4/10, train loss: 0.4810, train metric: 0.8301, valid metric: 0.8310
Epoch 5/10, train loss: 0.4558, train metric: 0.8404, valid metric: 0.8354
Epoch 6/10, train loss: 0.4388, train metric: 0.8461, valid metric: 0.8446
Epoch 7/10, train loss: 0.4240, train metric: 0.8512, valid metric: 0.8410
Epoch 8/10, train loss: 0.4123, train metric: 0.8564, valid metric: 0.8508
Epoch 9/10, train loss: 0.3997, train metric: 0.8599, valid metric: 0.8530


[I 2026-01-26 18:33:58,721] Trial 1 finished with value: 0.8543999791145325 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8543999791145325.


...

Epoch 10/10, train loss: 2.2414, train metric: 0.2299, valid metric: 0.2334
Epoch 1/10, train loss: 2.3035, train metric: 0.1373, valid metric: 0.1526
Epoch 2/10, train loss: 2.3005, train metric: 0.1569, valid metric: 0.1724
Epoch 3/10, train loss: 2.2975, train metric: 0.1755, valid metric: 0.1896
Epoch 4/10, train loss: 2.2945, train metric: 0.1941, valid metric: 0.2132
Epoch 5/10, train loss: 2.2914, train metric: 0.2105, valid metric: 0.2288
Epoch 6/10, train loss: 2.2884, train metric: 0.2261, valid metric: 0.2418
Epoch 7/10, train loss: 2.2853, train metric: 0.2419, valid metric: 0.2580
Epoch 8/10, train loss: 2.2823, train metric: 0.2581, valid metric: 0.2742
Epoch 9/10, train loss: 2.2792, train metric: 0.2736, valid metric: 0.2918


[I 2026-01-26 18:34:54,296] Trial 3 finished with value: 0.30959999561309814 and parameters: {'learning_rate': 1.7073967431528103e-05, 'n_hidden': 263}. Best is trial 1 with value: 0.8543999791145325.


Epoch 10/10, train loss: 2.2761, train metric: 0.2897, valid metric: 0.3096
Epoch 1/10, train loss: 1.8379, train metric: 0.4869, valid metric: 0.6208
Epoch 2/10, train loss: 0.9751, train metric: 0.6666, valid metric: 0.6978
Epoch 3/10, train loss: 0.7608, train metric: 0.7253, valid metric: 0.7416
Epoch 4/10, train loss: 0.6704, train metric: 0.7639, valid metric: 0.7720
Epoch 5/10, train loss: 0.6108, train metric: 0.7914, valid metric: 0.7906
Epoch 6/10, train loss: 0.5686, train metric: 0.8054, valid metric: 0.8052
Epoch 7/10, train loss: 0.5385, train metric: 0.8164, valid metric: 0.8082
Epoch 8/10, train loss: 0.5158, train metric: 0.8243, valid metric: 0.8212
Epoch 9/10, train loss: 0.4988, train metric: 0.8281, valid metric: 0.8222


[I 2026-01-26 18:35:22,727] Trial 4 finished with value: 0.8222000002861023 and parameters: {'learning_rate': 0.002537815508265664, 'n_hidden': 218}. Best is trial 1 with value: 0.8543999791145325.


Epoch 10/10, train loss: 0.4842, train metric: 0.8331, valid metric: 0.8092


In [96]:
study.best_params

{'learning_rate': 0.008471801418819975, 'n_hidden': 188}

In [97]:
study.best_value

0.8543999791145325

In [104]:
# Pruning

def objective(trial, train_loader, valid_loader):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10) #.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    accuracy = accuracy.to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(model, optimizer, xentropy, accuracy, train_loader,
                         valid_loader, n_epochs=1)
        validation_accuracy = max(history["valid_metrics"])
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy

In [105]:
objective_with_data = lambda trial: objective(
    trial, train_loader=train_loader, valid_loader=valid_loader)

In [106]:
torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler,
                            pruner=pruner)
study.optimize(objective_with_data, n_trials=20)

[I 2026-01-26 18:58:05,903] A new study created in memory with name: no-name-86f62772-6d58-451d-8748-546b22a0f00b


Epoch 1/1, train loss: 2.2769, train metric: 0.1471, valid metric: 0.1862
Epoch 1/1, train loss: 2.2093, train metric: 0.2794, valid metric: 0.3500
Epoch 1/1, train loss: 2.1164, train metric: 0.4110, valid metric: 0.4554
Epoch 1/1, train loss: 1.9776, train metric: 0.5137, valid metric: 0.5560
Epoch 1/1, train loss: 1.7867, train metric: 0.5826, valid metric: 0.6026
Epoch 1/1, train loss: 1.5775, train metric: 0.6184, valid metric: 0.6228
Epoch 1/1, train loss: 1.3978, train metric: 0.6288, valid metric: 0.6326
Epoch 1/1, train loss: 1.2605, train metric: 0.6360, valid metric: 0.6372
Epoch 1/1, train loss: 1.1572, train metric: 0.6467, valid metric: 0.6424
Epoch 1/1, train loss: 1.0782, train metric: 0.6537, valid metric: 0.6436
Epoch 1/1, train loss: 1.0162, train metric: 0.6611, valid metric: 0.6530
Epoch 1/1, train loss: 0.9665, train metric: 0.6689, valid metric: 0.6620
Epoch 1/1, train loss: 0.9258, train metric: 0.6761, valid metric: 0.6700
Epoch 1/1, train loss: 0.8919, train m

[I 2026-01-26 18:59:04,573] Trial 0 finished with value: 0.7089999914169312 and parameters: {'learning_rate': 0.00031489116479568613, 'n_hidden': 287}. Best is trial 0 with value: 0.7089999914169312.


Epoch 1/1, train loss: 0.7647, train metric: 0.7196, valid metric: 0.7082
Epoch 1/1, train loss: 1.1485, train metric: 0.6157, valid metric: 0.7332
Epoch 1/1, train loss: 0.6133, train metric: 0.7863, valid metric: 0.8082
Epoch 1/1, train loss: 0.5200, train metric: 0.8179, valid metric: 0.8136
Epoch 1/1, train loss: 0.4783, train metric: 0.8312, valid metric: 0.8234
Epoch 1/1, train loss: 0.4532, train metric: 0.8402, valid metric: 0.8028
Epoch 1/1, train loss: 0.4356, train metric: 0.8464, valid metric: 0.8444
Epoch 1/1, train loss: 0.4208, train metric: 0.8511, valid metric: 0.8274
Epoch 1/1, train loss: 0.4082, train metric: 0.8563, valid metric: 0.8392
Epoch 1/1, train loss: 0.3981, train metric: 0.8605, valid metric: 0.8528
Epoch 1/1, train loss: 0.3882, train metric: 0.8638, valid metric: 0.8582
Epoch 1/1, train loss: 0.3783, train metric: 0.8662, valid metric: 0.8538
Epoch 1/1, train loss: 0.3699, train metric: 0.8692, valid metric: 0.8566
Epoch 1/1, train loss: 0.3632, train m

[I 2026-01-26 19:00:01,071] Trial 1 finished with value: 0.8679999709129333 and parameters: {'learning_rate': 0.008471801418819975, 'n_hidden': 188}. Best is trial 1 with value: 0.8679999709129333.


...

Epoch 1/1, train loss: 1.5604, train metric: 0.5038, valid metric: 0.6536
Epoch 1/1, train loss: 0.6189, train metric: 0.7751, valid metric: 0.8314
Epoch 1/1, train loss: 0.4238, train metric: 0.8452, valid metric: 0.8488
Epoch 1/1, train loss: 0.3825, train metric: 0.8592, valid metric: 0.8550
Epoch 1/1, train loss: 0.3570, train metric: 0.8682, valid metric: 0.8598
Epoch 1/1, train loss: 0.3377, train metric: 0.8745, valid metric: 0.8656
Epoch 1/1, train loss: 0.3246, train metric: 0.8793, valid metric: 0.8438
Epoch 1/1, train loss: 0.3121, train metric: 0.8847, valid metric: 0.8676
Epoch 1/1, train loss: 0.3014, train metric: 0.8883, valid metric: 0.8732
Epoch 1/1, train loss: 0.2923, train metric: 0.8900, valid metric: 0.8812
Epoch 1/1, train loss: 0.2851, train metric: 0.8930, valid metric: 0.8782
Epoch 1/1, train loss: 0.2779, train metric: 0.8953, valid metric: 0.8784
Epoch 1/1, train loss: 0.2719, train metric: 0.8981, valid metric: 0.8666
Epoch 1/1, train loss: 0.2646, train m

[I 2026-01-26 19:09:42,593] Trial 18 finished with value: 0.885200023651123 and parameters: {'learning_rate': 0.09548128419071336, 'n_hidden': 50}. Best is trial 12 with value: 0.8871999979019165.


Epoch 1/1, train loss: 0.2348, train metric: 0.9110, valid metric: 0.8852


[I 2026-01-26 19:09:45,422] Trial 19 pruned. 


Epoch 1/1, train loss: 0.7417, train metric: 0.7395, valid metric: 0.7644


### Saving and Loading

In [107]:
torch.save(model, "my_fashion_mnist.pt")

In [108]:
loaded_model = torch.load("my_fashion_mnist.pt", weights_only=False)

In [109]:
loaded_model.eval()
y_pred_logits = loaded_model(X_new)
y_pred_logits

tensor([[-4.5526,  1.3295, -3.4186, -4.2696, -1.4952,  1.3122, -4.6409, 15.3466,
         -5.7657,  7.0867],
        [-1.1088, -4.0862,  9.6883,  0.9803, 11.6803, -8.8998,  6.6957, -7.9252,
         -2.2393, -5.5136],
        [ 0.2395, -4.4264,  6.5612,  0.0641,  3.4550, -2.6102,  7.7748, -3.3222,
         -3.3743, -4.1302]], grad_fn=<AddmmBackward0>)

In [110]:
# Save only the weights - recommended
torch.save(model.state_dict(), "my_fashion_mnist_weights.pt")

In [112]:
new_model = ImageClassifier(n_inputs=1 * 28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10)
loaded_weights = torch.load("my_fashion_mnist_weights.pt", weights_only=True)
new_model.load_state_dict(loaded_weights)
new_model.eval()
new_model(X_new)

tensor([[-4.5526,  1.3295, -3.4186, -4.2696, -1.4952,  1.3122, -4.6409, 15.3466,
         -5.7657,  7.0867],
        [-1.1088, -4.0862,  9.6883,  0.9803, 11.6803, -8.8998,  6.6957, -7.9252,
         -2.2393, -5.5136],
        [ 0.2395, -4.4264,  6.5612,  0.0641,  3.4550, -2.6102,  7.7748, -3.3222,
         -3.3743, -4.1302]], grad_fn=<AddmmBackward0>)

In [115]:
# save also the model architecture
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {
        "n_inputs": 1 * 28 * 28, 
        "n_hidden1": 300,
        "n_hidden2": 100,
        "n_classes": 10
        }
}
torch.save(model_data, "my_fashion_mnist_model.pt")

In [116]:
loaded_data = torch.load("my_fashion_mnist_model.pt", weights_only=True)
new_model = ImageClassifier(**loaded_data["model_hyperparameters"])
new_model.load_state_dict(loaded_data["model_state_dict"])
new_model.eval()
new_model(X_new)

tensor([[-4.5526,  1.3295, -3.4186, -4.2696, -1.4952,  1.3122, -4.6409, 15.3466,
         -5.7657,  7.0867],
        [-1.1088, -4.0862,  9.6883,  0.9803, 11.6803, -8.8998,  6.6957, -7.9252,
         -2.2393, -5.5136],
        [ 0.2395, -4.4264,  6.5612,  0.0641,  3.4550, -2.6102,  7.7748, -3.3222,
         -3.3743, -4.1302]], grad_fn=<AddmmBackward0>)

# Compiling and Optimizing

In [117]:
# Convert to torchscript
torchscript_model = torch.jit.trace(model, X_new)

In [118]:
# for dynamic models
torchscript_model = torch.jit.script(model)

In [119]:
optimized_model = torch.jit.optimize_for_inference(torchscript_model)

In [120]:
optimized_model.save("optimized_model.pt")

In [121]:
torchscript_model.save('my_fashion_mnist_torchscript.pt')

In [122]:
loaded_torchscript_model = torch.jit.load("my_fashion_mnist_torchscript.pt")

In [ ]:
# with compile
compiled_model = torch.compile(model)